# CS336 BPE Training on Colab High-RAM

This notebook trains the byte-level BPE tokenizer on TinyStories or OpenWebText from `data/`.

Recommended Colab setup:

- Runtime type: High-RAM. GPU is not useful for this CPU/text-processing job.
- For an 80GB RAM machine, start with `NUM_WORKERS=8` and `CHUNK_SIZE_MB=512`.
- Always run the smoke test first, then launch the full OWT run.
- Save outputs to Google Drive so a Colab disconnect does not lose finished tokenizer files.


In [ ]:
# Runtime sanity check
!python --version
!nproc
!free -h
!df -h /content || true


In [ ]:
# Optional but recommended: persist final vocab/merges/logs to Google Drive.
from pathlib import Path
import sys

USE_DRIVE = True
DRIVE_ROOT = None

if USE_DRIVE and "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")

print("DRIVE_ROOT =", DRIVE_ROOT)


## Get The Repo

If this notebook is already running from the repository root, the next cell will use it directly.
Otherwise, set `REPO_URL` to your GitHub repo URL before running the cell.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = ""  # Example: "https://github.com/<user>/cs336-assignment1-basics.git"
REPO_DIR = Path("/content/cs336-assignment1-basics")

cwd = Path.cwd()
if (cwd / "pyproject.toml").exists() and (cwd / "cs336_basics").exists():
    REPO_DIR = cwd
elif not (REPO_DIR / "pyproject.toml").exists():
    if not REPO_URL:
        raise RuntimeError("Set REPO_URL to your repo URL, or upload/open this notebook from the repo root.")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Repo root:", Path.cwd())
!git status --short --branch || true


In [ ]:
# Minimal dependencies for BPE training. This avoids installing the full project/Torch stack on Colab.
%pip install -q regex psutil

import os
import psutil

print("CPU count:", os.cpu_count())
print("Available RAM GiB:", round(psutil.virtual_memory().available / 2**30, 2))


## Download Data

Set `DATASET = "owt"` for the full OpenWebText sample, or `"tinystories"` for the smaller TinyStories run.
The script expects files under repo-local `data/`.

In [ ]:
from pathlib import Path
import subprocess

DATASET = "owt"  # "owt" or "tinystories"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

DATA_FILES = {
    "tinystories": DATA_DIR / "TinyStoriesV2-GPT4-train.txt",
    "owt": DATA_DIR / "owt_train.txt",
}

if DATASET == "tinystories" and not DATA_FILES[DATASET].exists():
    subprocess.run(
        [
            "wget",
            "-c",
            "-O",
            str(DATA_FILES[DATASET]),
            "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt",
        ],
        check=True,
    )
elif DATASET == "owt" and not DATA_FILES[DATASET].exists():
    gz_path = DATA_DIR / "owt_train.txt.gz"
    subprocess.run(
        [
            "wget",
            "-c",
            "-O",
            str(gz_path),
            "https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_train.txt.gz",
        ],
        check=True,
    )
    subprocess.run(["gunzip", "-f", str(gz_path)], check=True)

print("Training file:", DATA_FILES[DATASET])
!ls -lh data


## Training Parameters

For an 80GB high-RAM Colab instance, `8 x 512MiB` is a reasonable first full OWT setting.
If RAM pressure is high, lower `CHUNK_SIZE_MB` to `256` or `NUM_WORKERS` to `4`.

In [ ]:
import os
import psutil
from pathlib import Path

VOCAB_SIZE = 32_000 if DATASET == "owt" else 10_000
AVAILABLE_GIB = psutil.virtual_memory().available / 2**30
NUM_WORKERS = min(8, max(1, (os.cpu_count() or 2) - 1))
CHUNK_SIZE_MB = 512 if AVAILABLE_GIB >= 60 else 256
MERGE_PROGRESS_EVERY = 500

if DRIVE_ROOT is not None:
    OUTPUT_ROOT = DRIVE_ROOT / "cs336_bpe_outputs"
else:
    OUTPUT_ROOT = Path("data/tokenizers")

OUTPUT_DIR = OUTPUT_ROOT / f"{DATASET}_vocab{VOCAB_SIZE}"
LOG_DIR = OUTPUT_ROOT / "logs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("VOCAB_SIZE =", VOCAB_SIZE)
print("NUM_WORKERS =", NUM_WORKERS)
print("CHUNK_SIZE_MB =", CHUNK_SIZE_MB)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("LOG_DIR =", LOG_DIR)


In [ ]:
# Shared runner: streams output to the notebook and writes a persistent log file.
import os
import subprocess
import sys
import time
from pathlib import Path

def run_and_log(cmd: list[str], log_path: Path) -> None:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(Path.cwd()) + os.pathsep + env.get("PYTHONPATH", "")
    print("Running:", " ".join(cmd))
    print("Log:", log_path)
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}")


## Smoke Test

This trains on only the first 256MiB. It validates data paths, imports, multiprocessing, output writing, and tokenizer readback before spending time on the full corpus.

In [ ]:
RUN_SMOKE = True

if RUN_SMOKE:
    smoke_output = OUTPUT_ROOT / f"smoke_{DATASET}_vocab1000"
    smoke_log = LOG_DIR / f"smoke_{DATASET}_{int(time.time())}.log"
    smoke_cmd = [
        sys.executable,
        "-u",
        "scripts/train_bpe_full.py",
        "--dataset",
        DATASET,
        "--vocab-size",
        "1000",
        "--max-bytes",
        str(256 * 1024 * 1024),
        "--num-workers",
        str(min(NUM_WORKERS, 4)),
        "--chunk-size-mb",
        "64",
        "--output-dir",
        str(smoke_output),
        "--merge-progress-every",
        "100",
    ]
    run_and_log(smoke_cmd, smoke_log)
    print("Smoke output:", smoke_output)


## Full Training

Set `RUN_FULL_TRAIN = True` after the smoke test succeeds. OWT full training can take a long time because the merge loop is CPU-heavy even after pre-token counting finishes.

In [ ]:
RUN_FULL_TRAIN = False  # Change to True when ready.

if RUN_FULL_TRAIN:
    full_log = LOG_DIR / f"full_{DATASET}_vocab{VOCAB_SIZE}_{int(time.time())}.log"
    full_cmd = [
        sys.executable,
        "-u",
        "scripts/train_bpe_full.py",
        "--dataset",
        DATASET,
        "--vocab-size",
        str(VOCAB_SIZE),
        "--num-workers",
        str(NUM_WORKERS),
        "--chunk-size-mb",
        str(CHUNK_SIZE_MB),
        "--output-dir",
        str(OUTPUT_DIR),
        "--merge-progress-every",
        str(MERGE_PROGRESS_EVERY),
    ]
    run_and_log(full_cmd, full_log)
    print("Full output:", OUTPUT_DIR)
else:
    print("Full training is disabled. Set RUN_FULL_TRAIN = True in this cell when ready.")


## Validate The Saved Tokenizer

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from cs336_basics.tokenizer import Tokenizer

VOCAB_PATH = OUTPUT_DIR / "vocab.json"
MERGES_PATH = OUTPUT_DIR / "merges.txt"

if VOCAB_PATH.exists() and MERGES_PATH.exists():
    tokenizer = Tokenizer.from_files(VOCAB_PATH, MERGES_PATH, special_tokens=["<|endoftext|>"])
    text = "Hello from Colab.<|endoftext|>BPE training finished."
    ids = tokenizer.encode(text)
    print("num_ids:", len(ids))
    print("roundtrip_ok:", tokenizer.decode(ids) == text)
else:
    print("Tokenizer files are not present yet:")
    print(VOCAB_PATH)
    print(MERGES_PATH)


In [ ]:
# Inspect final artifacts.
!ls -lh "$OUTPUT_DIR" 2>/dev/null || true
!du -sh "$OUTPUT_DIR" 2>/dev/null || true
!ls -lh "$LOG_DIR" 2>/dev/null || true
